In [5]:
# Cell 1 — Install required libraries
!pip install transformers datasets evaluate accelerate -q

In [6]:
# Cell 2 — Load the AG News dataset
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")
print(dataset)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [7]:
# Cell 3 — Explore the dataset
# See the 4 category names
print("Label names:", dataset['train'].features['label'].names)
print()

# Look at 3 real examples
for i in range(3):
    example = dataset['train'][i]
    label_name = dataset['train'].features['label'].names[example['label']]
    print(f"Category: {label_name}")
    print(f"Text: {example['text']}")
    print("-" * 60)

Label names: ['World', 'Sports', 'Business', 'Sci/Tech']

Category: Business
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
------------------------------------------------------------
Category: Business
Text: Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
------------------------------------------------------------
Category: Business
Text: Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
------------------------------------------------------------


In [8]:
# Cell 4 — Load tokenizer and see how it works
from transformers import AutoTokenizer

# Load BERT's tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Try it on one real example from our dataset
sample_text = dataset['train'][0]['text']
print("Original text:")
print(sample_text)
print()

# Tokenize it
tokens = tokenizer(sample_text, truncation=True, max_length=128)

print("Token IDs (numbers BERT reads):")
print(tokens['input_ids'])
print()

print("Decoded back to words:")
print(tokenizer.decode(tokens['input_ids']))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original text:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Token IDs (numbers BERT reads):
[101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813, 2395, 1005, 1055, 1040, 11101, 2989, 1032, 2316, 1997, 11087, 1011, 22330, 8713, 2015, 1010, 2024, 3773, 2665, 2153, 1012, 102]

Decoded back to words:
[CLS] wall st. bears claw back into the black ( reuters ) reuters - short - sellers, wall street ' s dwindling \ band of ultra - cynics, are seeing green again. [SEP]


In [9]:
# Cell 5 — Tokenize the entire dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",  # short sentences → pad with zeros to same length
        truncation=True,       # long sentences → cut at 128 tokens
        max_length=128         # 128 is enough for news headlines
    )

# Apply tokenizer to every row in the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Done! Tokenized dataset:")
print(tokenized_dataset)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Done! Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})


In [10]:
# Cell 6 — Load BERT model for classification
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4        # tell BERT: there are 4 categories to choose from
)

print("Model loaded successfully!")
print(f"Total parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Total parameters: 109,485,316


In [11]:
# Cell 7 — Set up training configuration
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# Load accuracy metric
metric = evaluate.load("accuracy")

# This function runs after each epoch to check performance
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)  # pick the highest score
    return metric.compute(predictions=predictions, references=labels)

# All training settings in one place
training_args = TrainingArguments(
    output_dir="./bert-ag-news",       # where to save the model
    num_train_epochs=1,                # go through training data once
    per_device_train_batch_size=16,    # process 16 examples at a time
    per_device_eval_batch_size=32,
    eval_strategy="epoch",             # check accuracy after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"                   # don't send logs anywhere
)

# Use small subset — 5000 train, 1000 test (full dataset = 2+ hours!)
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
small_test  = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

# Create the Trainer — it manages everything automatically
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

print("Trainer is ready!")
print(f"Training on {len(small_train)} examples")
print(f"Evaluating on {len(small_test)} examples")

Trainer is ready!
Training on 5000 examples
Evaluating on 1000 examples


In [12]:
!pip install evaluate -q

In [13]:
# Cell 9 — Train the model!
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.314916,0.282594,0.902000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=313, training_loss=0.39834368647858737, metrics={'train_runtime': 135.0614, 'train_samples_per_second': 37.02, 'train_steps_per_second': 2.317, 'total_flos': 328894725120000.0, 'train_loss': 0.39834368647858737, 'epoch': 1.0})

In [14]:
# Cell 10 — Evaluate with Accuracy + F1 score
from sklearn.metrics import classification_report
import numpy as np

# Get predictions on test set
predictions = trainer.predict(small_test)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Category names
label_names = ['World', 'Sports', 'Business', 'Sci/Tech']

# Full report
print(classification_report(true_labels, pred_labels, target_names=label_names))

              precision    recall  f1-score   support

       World       0.95      0.86      0.90       266
      Sports       0.95      0.98      0.97       246
    Business       0.90      0.83      0.87       246
    Sci/Tech       0.82      0.93      0.87       242

    accuracy                           0.90      1000
   macro avg       0.90      0.90      0.90      1000
weighted avg       0.91      0.90      0.90      1000



In [15]:
# Cell 11 — Test with your own sentences!
from transformers import pipeline

# Create a ready-to-use classifier
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

# Label mapping
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# Test sentences — feel free to change these!
test_sentences = [
    "Pakistan wins the cricket World Cup in a thrilling final against India",
    "Apple launches new iPhone with AI features built directly into the chip",
    "Stock markets crash as inflation fears grow across Europe",
    "United Nations calls emergency meeting over conflict in Middle East"
]

print("Testing BERT on custom sentences:\n")
for sentence in test_sentences:
    result = classifier(sentence)[0]
    label_id = int(result['label'].split('_')[-1])
    category = id2label[label_id]
    confidence = result['score'] * 100
    print(f"Text: {sentence}")
    print(f"Prediction: {category} ({confidence:.1f}% confident)")
    print("-" * 60)

Testing BERT on custom sentences:

Text: Pakistan wins the cricket World Cup in a thrilling final against India
Prediction: Sports (78.7% confident)
------------------------------------------------------------
Text: Apple launches new iPhone with AI features built directly into the chip
Prediction: Sci/Tech (97.1% confident)
------------------------------------------------------------
Text: Stock markets crash as inflation fears grow across Europe
Prediction: Business (93.8% confident)
------------------------------------------------------------
Text: United Nations calls emergency meeting over conflict in Middle East
Prediction: World (99.0% confident)
------------------------------------------------------------


In [17]:
# Cell 12 — Install Gradio
!pip install gradio -q

In [18]:
# Cell 13 — Deploy with Gradio
import gradio as gr

# Label mapping
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# This function runs every time someone types a headline
def predict_news(headline):
    result = classifier(headline)[0]
    label_id = int(result['label'].split('_')[-1])
    category = id2label[label_id]
    confidence = result['score'] * 100

    # Return a nicely formatted result
    return {
        cat: (result['score'] if cat == category else 0.0)
        for cat in id2label.values()
    }

# Build the web interface
demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Type any news headline here...",
        label="News Headline"
    ),
    outputs=gr.Label(
        num_top_classes=4,
        label="Category Prediction"
    ),
    title="📰 News Topic Classifier",
    description="Powered by BERT — type any news headline and AI will classify it!",
    examples=[
        ["Pakistan beats India in T20 World Cup final"],
        ["NASA discovers water on Mars surface"],
        ["Federal Reserve raises interest rates by 0.5%"],
        ["UN Security Council meets over Gaza conflict"]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d3b2847edf467fa46.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
